In [ ]:
# ============================================================
# BRANCH B — STRUCTURAL CONVERSION
# D8 — World Bank — Bhutan Land Management Project
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch B: Structural Conversion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

!apt-get -qq update
!apt-get -qq install -y libreoffice-writer antiword > /dev/null
!pip -q install pymupdf pymupdf4llm

from google.colab import files
from pathlib import Path
from collections import Counter

import hashlib
import json
import math
import shutil
import subprocess

import fitz
import pandas as pd
import pymupdf4llm

DOCUMENT_ID = "D8"
DOCUMENT_NAME = (
    "World Bank — Bhutan - Land Management Project — "
    "Project Information Document (PID), Concept Stage"
)

BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

CONVERSION_METHOD = (
    "LibreOffice DOC-to-PDF conversion followed by "
    "pymupdf4llm page-aware Markdown conversion and "
    "deterministic wrapped-label structural reconstruction"
)

LLM_INPUT_REPRESENTATION = "Structural Markdown"

SOURCE_FORMAT = ".doc"
EXPECTED_SOURCE_SHA256 = "61aacfd3138ecfba59fac51d29a970de45d8a909c74b744e756e8a283666c7b5"
EXPECTED_PHYSICAL_PAGE_COUNT = 4

EXPECTED_RECORD_COUNT = 49

EXPECTED_CATEGORY_COUNTS = {
    "Project metadata": 13,
    "Development issue": 10,
    "Bank rationale": 2,
    "Project objective": 3,
    "Project component": 3,
    "Safeguard policy": 6,
    "Financing": 7,
    "Contact information": 5
}

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location"
]

STRING_OR_NULL_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location"
]

MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Source Location"
]

ALLOWED_CATEGORIES = set(EXPECTED_CATEGORY_COUNTS)

EXPECTED_QUALIFIER_VALUES = {
    "at least",
    "another",
    "less than",
    "some",
    "up to"
}

OUTPUT_DIR = Path("outputs_D8_branch_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONVERSION_DIR = OUTPUT_DIR / "conversion"
CONVERSION_DIR.mkdir(parents=True, exist_ok=True)

INTERMEDIATE_PDF_PATH = CONVERSION_DIR / "D8_branch_B_intermediate.pdf"
STRUCTURAL_MARKDOWN_PATH = OUTPUT_DIR / "D8_branch_B_structural_markdown.md"
CONVERSION_REPORT_PATH = OUTPUT_DIR / "D8_branch_B_conversion_report.json"
CONVERSION_INTEGRITY_PATH = OUTPUT_DIR / "D8_branch_B_conversion_integrity.json"
REPRESENTATION_METADATA_PATH = OUTPUT_DIR / "D8_branch_B_representation.json"
PROMPT_PATH = OUTPUT_DIR / "D8_branch_B_prompt.txt"
EXPERIMENT_METADATA_PRE_PATH = OUTPUT_DIR / "D8_branch_B_experiment_metadata_pre.json"
RAW_RESPONSE_PATH = OUTPUT_DIR / "D8_branch_B_raw_response.txt"
PARSED_EXTRACTION_PATH = OUTPUT_DIR / "D8_branch_B_parsed_extraction.json"
TECHNICAL_DIAGNOSTICS_PATH = OUTPUT_DIR / "D8_branch_B_technical_diagnostics.json"
EXPERIMENT_METADATA_PATH = OUTPUT_DIR / "D8_branch_B_experiment_metadata.json"
EXPERIMENT_SUMMARY_PATH = OUTPUT_DIR / "D8_branch_B_experiment_summary.json"

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Expected physical pages:", EXPECTED_PHYSICAL_PAGE_COUNT)
print("Frozen Stage 1 records:", EXPECTED_RECORD_COUNT)
print("Expected fields:", len(EXPECTED_FIELDS))


In [ ]:
# ============================================================
# 1. Upload and verify the exact original D8 legacy DOC
# ============================================================

uploaded = files.upload()

doc_paths = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".doc")
]

if len(doc_paths) != 1:
    raise ValueError("Upload exactly one original D8 legacy .doc file.")

SOURCE_PATH = doc_paths[0]

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

SOURCE_SHA256 = sha256_file(SOURCE_PATH)
SOURCE_HASH_MATCH = SOURCE_SHA256 == EXPECTED_SOURCE_SHA256

if SOURCE_PATH.suffix.lower() != SOURCE_FORMAT:
    raise ValueError("Unexpected D8 source format.")

if not SOURCE_HASH_MATCH:
    raise ValueError(
        "Uploaded D8 DOC does not match the frozen Stage 1 source identity."
    )

source_bytes = SOURCE_PATH.read_bytes()

OLE_SIGNATURE = bytes.fromhex("D0CF11E0A1B11AE1")
LEGACY_BINARY_DOC = source_bytes.startswith(OLE_SIGNATURE)

if not LEGACY_BINARY_DOC:
    raise ValueError(
        "The uploaded D8 file does not appear to be the expected legacy binary DOC."
    )

print("Source file:", SOURCE_PATH.name)
print("Source SHA-256:", SOURCE_SHA256)
print("Frozen source match:", SOURCE_HASH_MATCH)
print("Legacy binary DOC signature:", LEGACY_BINARY_DOC)


In [ ]:
# ============================================================
# 2. Diagnostic source-text extraction with antiword
# ============================================================

antiword_result = subprocess.run(
    ["antiword", str(SOURCE_PATH)],
    capture_output=True,
    text=True,
    errors="replace"
)

if antiword_result.returncode != 0:
    raise RuntimeError(
        "antiword could not recover the D8 machine-readable source text.\n"
        + antiword_result.stderr
    )

SOURCE_DIAGNOSTIC_TEXT = antiword_result.stdout

if not SOURCE_DIAGNOSTIC_TEXT.strip():
    raise ValueError("antiword recovered an empty D8 source text.")

REQUIRED_SECTION_MARKERS = {
    "header": "PROJECT INFORMATION DOCUMENT (PID)",
    "concept_stage": "CONCEPT STAGE",
    "development_issues":
        "1. Key development issues and rationale for Bank involvement",
    "objectives": "2. Proposed objective(s)",
    "description": "3. Preliminary description",
    "safeguards": "4. Safeguard Policies that Might Apply",
    "financing": "5. Tentative financing",
    "contact": "6. Contact point"
}

REQUIRED_SOURCE_MARKERS = [
    "AB526",
    "P087039",
    "Indigenous Pelple",
    "BORROWER/RECEPIENT",
    "F1",
    "one-quarter",
    "one-third",
    "16.5"
]

section_marker_checks = {
    key: marker.casefold() in SOURCE_DIAGNOSTIC_TEXT.casefold()
    for key, marker in REQUIRED_SECTION_MARKERS.items()
}

source_marker_checks = {
    marker: marker.casefold() in SOURCE_DIAGNOSTIC_TEXT.casefold()
    for marker in REQUIRED_SOURCE_MARKERS
}

SOURCE_TEXT_DIAGNOSTICS = {
    "machine_readable_text_recovered": True,
    "antiword_used_for_diagnostics_only": True,
    "antiword_text_used_as_model_input": False,
    "section_marker_checks": section_marker_checks,
    "all_section_markers_present": all(section_marker_checks.values()),
    "source_marker_checks": source_marker_checks,
    "all_source_markers_present": all(source_marker_checks.values()),
    "character_count": len(SOURCE_DIAGNOSTIC_TEXT),
    "word_count": len(SOURCE_DIAGNOSTIC_TEXT.split())
}

print(json.dumps(SOURCE_TEXT_DIAGNOSTICS, indent=2, ensure_ascii=False))

if not all(section_marker_checks.values()):
    raise ValueError("Expected D8 section markers were not all recovered.")

if not all(source_marker_checks.values()):
    raise ValueError("Expected D8 source-fidelity markers were not all recovered.")


In [ ]:
# ============================================================
# 3. Deterministically render the complete legacy DOC to PDF
# ============================================================

if INTERMEDIATE_PDF_PATH.exists():
    INTERMEDIATE_PDF_PATH.unlink()

conversion_command = [
    "libreoffice",
    "--headless",
    "--convert-to", "pdf",
    "--outdir", str(CONVERSION_DIR),
    str(SOURCE_PATH)
]

conversion_result = subprocess.run(
    conversion_command,
    capture_output=True,
    text=True
)

if conversion_result.returncode != 0:
    raise RuntimeError(
        "LibreOffice DOC-to-PDF conversion failed.\n"
        + conversion_result.stderr
    )

generated_pdf = CONVERSION_DIR / f"{SOURCE_PATH.stem}.pdf"

if not generated_pdf.exists():
    pdf_candidates = list(CONVERSION_DIR.glob("*.pdf"))

    if len(pdf_candidates) != 1:
        raise FileNotFoundError(
            "Could not unambiguously locate the rendered D8 PDF."
        )

    generated_pdf = pdf_candidates[0]

if generated_pdf.resolve() != INTERMEDIATE_PDF_PATH.resolve():
    shutil.move(
        str(generated_pdf),
        str(INTERMEDIATE_PDF_PATH)
    )

INTERMEDIATE_PDF_SHA256 = sha256_file(INTERMEDIATE_PDF_PATH)

pdf_document = fitz.open(INTERMEDIATE_PDF_PATH)
OBSERVED_PAGE_COUNT = len(pdf_document)
PAGE_COUNT_VALID = OBSERVED_PAGE_COUNT == EXPECTED_PHYSICAL_PAGE_COUNT

if not PAGE_COUNT_VALID:
    raise ValueError(
        f"Expected {EXPECTED_PHYSICAL_PAGE_COUNT} rendered pages; "
        f"observed {OBSERVED_PAGE_COUNT}."
    )

rendered_page_text_counts = [
    len(page.get_text("text").strip())
    for page in pdf_document
]

if not all(count > 0 for count in rendered_page_text_counts):
    raise ValueError(
        "One or more rendered D8 PDF pages do not contain machine-readable text."
    )

print("Intermediate PDF:", INTERMEDIATE_PDF_PATH)
print("Intermediate PDF SHA-256:", INTERMEDIATE_PDF_SHA256)
print("Rendered pages:", OBSERVED_PAGE_COUNT)
print("Characters by rendered page:", rendered_page_text_counts)


In [ ]:
# ============================================================
# 4. Convert the complete rendered PDF to page-aware Markdown
#    + deterministic financing-label structural reconstruction
# ============================================================

try:
    page_chunks = pymupdf4llm.to_markdown(
        str(INTERMEDIATE_PDF_PATH),
        page_chunks=True,
        write_images=False,
        show_progress=True
    )
except Exception as exc:
    raise RuntimeError(
        "D8 page-aware Markdown conversion failed. "
        "No fallback representation is used. "
        f"Original error: {exc}"
    )


# ------------------------------------------------------------
# 4.1 Page-chunk output
# ------------------------------------------------------------

if not isinstance(page_chunks, list):
    raise TypeError(
        "Expected pymupdf4llm page_chunks=True "
        "to return a list."
    )

if len(page_chunks) != EXPECTED_PHYSICAL_PAGE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_PHYSICAL_PAGE_COUNT} "
        f"Markdown page chunks; "
        f"observed {len(page_chunks)}."
    )


# ------------------------------------------------------------
# 4.2 Recover converted Markdown for every physical page
# ------------------------------------------------------------

page_markdown = {}

for page_number, chunk in enumerate(
    page_chunks,
    start=1
):
    text = (
        chunk.get("text", "")
        if isinstance(chunk, dict)
        else str(chunk)
    )

    if not text.strip():
        raise ValueError(
            "Converted Markdown for "
            f"D8 source page {page_number} is empty."
        )

    page_markdown[page_number] = text.rstrip()


# ------------------------------------------------------------
# 4.3 Deterministically reconstruct financing labels that were
#     split across visual lines in the legacy source
# ------------------------------------------------------------

FINANCING_WRAPPED_LABEL_PREFIXES = {
    "LOCAL GOVTS. (PROV., DISTRICT, CITY) OF BORROWING",
    "NON-GOVERNMENT ORGANIZATION (NGO) OF BORROWING"
}


def parse_markdown_table_row(line):
    """
    Return Markdown table cells for a simple pipe-delimited row.
    Return None when the line is not a table row.
    """

    stripped = line.strip()

    if not (
        stripped.startswith("|")
        and stripped.endswith("|")
    ):
        return None

    cells = [
        cell.strip()
        for cell in stripped[1:-1].split("|")
    ]

    return cells


def rebuild_markdown_table_row(cells):
    return (
        "| "
        + " | ".join(str(cell) for cell in cells)
        + " |"
    )


def reconstruct_financing_wrapped_labels(
    markdown_text
):
    """
    Reattach a standalone COUNTRY continuation row to one of
    the two known financing labels immediately preceding it.

    This operates only when:
    1. the current row label is exactly COUNTRY;
    2. its value cell is empty;
    3. the immediately preceding row label is one of the
       source-grounded wrapped financing prefixes.
    """

    lines = markdown_text.splitlines()

    reconstructed_lines = []
    reconstruction_events = []

    for line_index, line in enumerate(lines):

        current_cells = parse_markdown_table_row(
            line
        )

        is_country_continuation = False

        if (
            current_cells is not None
            and len(current_cells) >= 2
        ):
            current_label = current_cells[0].strip()

            remaining_cells = [
                cell.strip()
                for cell in current_cells[1:]
            ]

            is_country_continuation = (
                current_label == "COUNTRY"
                and all(
                    cell == ""
                    for cell in remaining_cells
                )
            )

        if (
            is_country_continuation
            and reconstructed_lines
        ):
            previous_line = reconstructed_lines[-1]

            previous_cells = (
                parse_markdown_table_row(
                    previous_line
                )
            )

            if (
                previous_cells is not None
                and len(previous_cells) >= 2
            ):
                previous_label = (
                    previous_cells[0].strip()
                )

                if (
                    previous_label
                    in FINANCING_WRAPPED_LABEL_PREFIXES
                ):
                    previous_cells[0] = (
                        previous_label
                        + " COUNTRY"
                    )

                    reconstructed_lines[-1] = (
                        rebuild_markdown_table_row(
                            previous_cells
                        )
                    )

                    reconstruction_events.append({
                        "source_line_index":
                            line_index,
                        "continuation":
                            "COUNTRY",
                        "original_prefix":
                            previous_label,
                        "reconstructed_label":
                            previous_cells[0]
                    })

                    continue

        reconstructed_lines.append(line)

    reconstructed_text = (
        "\n".join(reconstructed_lines)
        .rstrip()
        + "\n"
    )

    return (
        reconstructed_text,
        reconstruction_events
    )

(
    page_markdown[4],
    FINANCING_LABEL_RECONSTRUCTION_EVENTS
) = reconstruct_financing_wrapped_labels(
    page_markdown[4]
)

FINANCING_LABEL_RECONSTRUCTION_COUNT = len(
    FINANCING_LABEL_RECONSTRUCTION_EVENTS
)

print(
    "Financing wrapped-label reconstructions:",
    FINANCING_LABEL_RECONSTRUCTION_COUNT
)

print(
    json.dumps(
        FINANCING_LABEL_RECONSTRUCTION_EVENTS,
        indent=2,
        ensure_ascii=False
    )
)


# ------------------------------------------------------------
# 4.4 Expected two structural reconstruction events
# ------------------------------------------------------------

if FINANCING_LABEL_RECONSTRUCTION_COUNT != 2:
    raise ValueError(
        "Expected exactly 2 D8 financing-label "
        "continuation reconstructions, but observed "
        f"{FINANCING_LABEL_RECONSTRUCTION_COUNT}."
    )

# ------------------------------------------------------------
# 4.5 Complete page-aware Markdown representation
# ------------------------------------------------------------

markdown_parts = [
    "# D8 — World Bank Bhutan Land Management Project PID",
    "",
    (
        "> Complete structural conversion of the "
        "original four-page legacy DOC."
    ),
    (
        "> No source pages have been removed."
    ),
    ""
]


for page_number in range(
    1,
    EXPECTED_PHYSICAL_PAGE_COUNT + 1
):
    markdown_parts.extend([
        f"## Source Page {page_number}",
        "",
        page_markdown[page_number],
        ""
    ])


STRUCTURAL_MARKDOWN = (
    "\n".join(markdown_parts)
    .rstrip()
    + "\n"
)

# ------------------------------------------------------------
# 4.6 Final Branch B representation
# ------------------------------------------------------------

STRUCTURAL_MARKDOWN_PATH.write_text(
    STRUCTURAL_MARKDOWN,
    encoding="utf-8"
)

STRUCTURAL_MARKDOWN_SHA256 = sha256_file(
    STRUCTURAL_MARKDOWN_PATH
)


print(
    "Structural Markdown:",
    STRUCTURAL_MARKDOWN_PATH
)

print(
    "Representation SHA-256:",
    STRUCTURAL_MARKDOWN_SHA256
)

print(
    "Converted pages:",
    len(page_markdown)
)

print(
    "Characters:",
    len(STRUCTURAL_MARKDOWN)
)

In [ ]:
# ============================================================
# 5. Conversion-integrity verification
# ============================================================

# ------------------------------------------------------------
# 5.1 Normalised copy used for integrity diagnostics
# ------------------------------------------------------------

def normalise_integrity_text(text):
    text = str(text).casefold()

    for char in [
        "*",
        "_",
        "`"
    ]:
        text = text.replace(
            char,
            ""
        )

    text = " ".join(
        text.split()
    )

    return text


normalised_markdown = normalise_integrity_text(
    STRUCTURAL_MARKDOWN
)

# ------------------------------------------------------------
# 5.2 Page-boundary checks
# ------------------------------------------------------------

page_boundary_checks = {
    str(page_number):
        f"## Source Page {page_number}"
        in STRUCTURAL_MARKDOWN
    for page_number
    in range(
        1,
        EXPECTED_PHYSICAL_PAGE_COUNT + 1
    )
}

# ------------------------------------------------------------
# 5.3 Section-preservation checks
# ------------------------------------------------------------

section_preservation_checks = {
    key:
        normalise_integrity_text(marker)
        in normalised_markdown

    for key, marker
    in REQUIRED_SECTION_MARKERS.items()
}


# ------------------------------------------------------------
# 5.4 Source-fidelity checks
# ------------------------------------------------------------

source_fidelity_checks = {
    marker:
        normalise_integrity_text(marker)
        in normalised_markdown

    for marker
    in REQUIRED_SOURCE_MARKERS
}


# ------------------------------------------------------------
# 5.5 Representative-content checks
# ------------------------------------------------------------

REPRESENTATIVE_CONTENT = [
    "60%",
    "520",
    "6.7%",
    "10%",
    "40%",
    "$0.50 million",
    "$10.0 million",
    "$6.0",
    "OP4.01",
    "OD4.20",
    "16.5",
    "Ai Chin Wee",
    "(202) 458-5049",
    "Awee@worldbank.org"
]


representative_content_checks = {
    marker:
        normalise_integrity_text(marker)
        in normalised_markdown

    for marker
    in REPRESENTATIVE_CONTENT
}


# ------------------------------------------------------------
# 5.6 Financing-table technical diagnostics
# ------------------------------------------------------------

EXPECTED_RECONSTRUCTED_FINANCING_LABELS = [
    (
        "LOCAL GOVTS. (PROV., DISTRICT, CITY) "
        "OF BORROWING COUNTRY"
    ),
    (
        "NON-GOVERNMENT ORGANIZATION (NGO) "
        "OF BORROWING COUNTRY"
    )
]


financing_label_checks = {
    label:
        normalise_integrity_text(label)
        in normalised_markdown

    for label
    in EXPECTED_RECONSTRUCTED_FINANCING_LABELS
}


standalone_country_row_found = False

for line in STRUCTURAL_MARKDOWN.splitlines():

    cells = parse_markdown_table_row(
        line
    )

    if (
        cells is not None
        and len(cells) >= 2
        and cells[0].strip() == "COUNTRY"
        and all(
            cell.strip() == ""
            for cell in cells[1:]
        )
    ):
        standalone_country_row_found = True
        break


financing_structure_checks = {
    "expected_reconstructed_labels_present":
        all(
            financing_label_checks.values()
        ),

    "exactly_two_reconstruction_events":
        (
            FINANCING_LABEL_RECONSTRUCTION_COUNT
            == 2
        ),

    "no_standalone_empty_country_row":
        not standalone_country_row_found
}


FINANCING_STRUCTURALLY_EVALUABLE = all(
    financing_structure_checks.values()
)


# ------------------------------------------------------------
# 5.7 Conversion report
# ------------------------------------------------------------

CONVERSION_REPORT = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "conversion_path":
        (
            "legacy DOC -> PDF -> page-aware Markdown "
            "-> deterministic wrapped-label reconstruction"
        ),

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "intermediate_pdf_file":
        INTERMEDIATE_PDF_PATH.name,

    "intermediate_pdf_sha256":
        INTERMEDIATE_PDF_SHA256,

    "representation_file":
        STRUCTURAL_MARKDOWN_PATH.name,

    "representation_sha256":
        STRUCTURAL_MARKDOWN_SHA256,

    "source_page_count_expected":
        EXPECTED_PHYSICAL_PAGE_COUNT,

    "rendered_pdf_page_count":
        OBSERVED_PAGE_COUNT,

    "markdown_page_count":
        len(page_markdown),

    "financing_label_reconstruction_count":
        FINANCING_LABEL_RECONSTRUCTION_COUNT,

    "financing_label_reconstruction_events":
        FINANCING_LABEL_RECONSTRUCTION_EVENTS,

    "conversion_fallback_used":
        False
}


CONVERSION_REPORT_PATH.write_text(
    json.dumps(
        CONVERSION_REPORT,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 5.8 Conversion-integrity object
# ------------------------------------------------------------

CONVERSION_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "source_sha256":
        SOURCE_SHA256,

    "source_hash_matches_frozen_identity":
        SOURCE_HASH_MATCH,

    "legacy_binary_doc_verified":
        LEGACY_BINARY_DOC,

    "expected_physical_page_count":
        EXPECTED_PHYSICAL_PAGE_COUNT,

    "rendered_pdf_page_count":
        OBSERVED_PAGE_COUNT,

    "converted_markdown_page_count":
        len(page_markdown),

    "page_count_valid":
        PAGE_COUNT_VALID,

    "all_source_pages_retained":
        all(
            page_boundary_checks.values()
        ),

    "page_boundary_checks":
        page_boundary_checks,

    "section_preservation_checks":
        section_preservation_checks,

    "all_expected_sections_preserved":
        all(
            section_preservation_checks.values()
        ),

    "source_fidelity_checks":
        source_fidelity_checks,

    "all_source_fidelity_markers_preserved":
        all(
            source_fidelity_checks.values()
        ),

    "representative_content_checks":
        representative_content_checks,

    "all_representative_content_preserved":
        all(
            representative_content_checks.values()
        ),

    "financing_label_checks":
        financing_label_checks,

    "financing_structure_checks":
        financing_structure_checks,

    "financing_structurally_evaluable":
        FINANCING_STRUCTURALLY_EVALUABLE,

    "financing_label_reconstruction_count":
        FINANCING_LABEL_RECONSTRUCTION_COUNT,

    "financing_label_reconstruction_events":
        FINANCING_LABEL_RECONSTRUCTION_EVENTS,

    "conversion_method":
        CONVERSION_METHOD,

    "conversion_fallback_used":
        False,

    "complete_source_document_retained":
        True,

    "scope_filtering_applied":
        False,

    "ocr_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "source_spelling_correction_applied":
        False,

    "deterministic_structural_reconstruction_applied":
        True,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "normalisation_applied":
        False,

    "conversion_integrity_passed":
        all([
            SOURCE_HASH_MATCH,
            LEGACY_BINARY_DOC,
            PAGE_COUNT_VALID,

            len(page_markdown)
            == EXPECTED_PHYSICAL_PAGE_COUNT,

            all(
                page_boundary_checks.values()
            ),

            all(
                section_preservation_checks.values()
            ),

            all(
                source_fidelity_checks.values()
            ),

            all(
                representative_content_checks.values()
            ),

            FINANCING_STRUCTURALLY_EVALUABLE
        ])
}


# ------------------------------------------------------------
# 5.9 Save integrity report
# ------------------------------------------------------------

CONVERSION_INTEGRITY_PATH.write_text(
    json.dumps(
        CONVERSION_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 5.10 Display integrity result
# ------------------------------------------------------------

print(
    json.dumps(
        CONVERSION_INTEGRITY,
        indent=2,
        ensure_ascii=False
    )
)


# ------------------------------------------------------------
# 5.11 Stop if conversion integrity failed
# ------------------------------------------------------------

if not CONVERSION_INTEGRITY[
    "conversion_integrity_passed"
]:
    raise ValueError(
        "D8 Branch B conversion integrity failed."
    )

In [ ]:
# ============================================================
# 6. Preserve Branch B representation metadata
# ============================================================

REPRESENTATION_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "representation_type":
        (
            "Complete legacy DOC rendered to PDF and "
            "converted to page-aware structural Markdown"
        ),

    "source_file":
        SOURCE_PATH.name,

    "source_format":
        SOURCE_FORMAT,

    "source_sha256":
        SOURCE_SHA256,

    "representation_file":
        STRUCTURAL_MARKDOWN_PATH.name,

    "representation_sha256":
        STRUCTURAL_MARKDOWN_SHA256,

    "intermediate_pdf_file":
        INTERMEDIATE_PDF_PATH.name,

    "intermediate_pdf_sha256":
        INTERMEDIATE_PDF_SHA256,

    "branch_name":
        BRANCH_NAME,

    "conversion_method":
        CONVERSION_METHOD,

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "source_page_count":
        EXPECTED_PHYSICAL_PAGE_COUNT,

    "converted_page_count":
        len(page_markdown),

    "structural_conversion_applied":
        True,

    "conversion_path":
        (
            "legacy DOC -> PDF -> page-aware Markdown "
            "-> deterministic wrapped-label reconstruction"
        ),

    "conversion_fallback_used":
        False,

    "complete_source_document_retained":
        True,

    "page_boundaries_made_explicit":
        True,

    "scope_enforced_by_prompt_not_representation_filtering":
        True,

    "antiword_used_for_diagnostics_only":
        True,

    "antiword_text_used_as_model_input":
        False,

    # --------------------------------------------------------
    # D8-specific deterministic structural reconstruction
    # --------------------------------------------------------

    "deterministic_structural_reconstruction_applied":
        True,

    "deterministic_structural_reconstruction_type":
        (
            "Reattachment of visually wrapped COUNTRY "
            "continuations to two financing-source labels"
        ),

    "financing_label_reconstruction_count":
        FINANCING_LABEL_RECONSTRUCTION_COUNT,

    "financing_label_reconstruction_events":
        FINANCING_LABEL_RECONSTRUCTION_EVENTS,

    # --------------------------------------------------------
    # Explicitly excluded operations
    # --------------------------------------------------------

    "ocr_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "source_spelling_correction_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "conversion_integrity_passed":
        CONVERSION_INTEGRITY[
            "conversion_integrity_passed"
        ]
}


REPRESENTATION_METADATA_PATH.write_text(
    json.dumps(
        REPRESENTATION_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        REPRESENTATION_METADATA,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 7. Create controlled Branch B extraction prompt
# ============================================================

BRANCH_B_PROMPT = """You are an information extraction assistant.

Extract the project-information records represented within the defined
scope of the attached structurally converted Markdown representation
of the original legacy Word document:

“Bhutan - Land Management Project”
Project Information Document (PID), Concept Stage.

Treat the attached structurally converted Markdown document as the only
source of information.

Include records from the following defined source regions.

1. Project metadata

Extract one record for each of these labelled header fields:

- Report No.
- Project Name
- Region
- Sector
- Project ID
- GEF Focal Area
- Borrower(s)
- Implementing Agency
- Environment Category
- Safeguard Classification
- Date PID Prepared
- Estimated Date of Appraisal Authorization
- Estimated Date of Board Approval

For the Sector field, preserve the complete represented sector text as
a single Value. Do not split its embedded percentages into additional
records.

For checkbox fields, extract the selected category represented by the
document.

Do not create separate records from implementing-agency telephone
numbers embedded within the Implementing Agency field.

2. Development issues

Within Section 1, “Key development issues and rationale for Bank
involvement”, extract the predefined quantitative development
observations concerning:

- the long-term forest-cover policy requirement;
- the share of country area set aside for protected areas;
- the additional area offered for wildlife corridors;
- population density per square kilometre of arable land;
- the urban growth rate;
- arable land as a share of land area;
- agricultural land affected by water erosion;
- the contribution of hydropower revenue to the development budget;
- villages not connected to feeder roads;
- villages facing food insecurity.

Do not extract comparison values concerning other world regions as
separate observations.

3. Bank rationale

Within the “Rationale for Bank Involvement” subsection, extract the
principal qualitative records concerning:

- limitations of the existing sector-oriented institutional framework
  in providing cross-sectoral accountability and incentive mechanisms;
- the governance, political-will and environmental-stewardship factors
  supporting Bhutan's suitability for GEF support.

Do not create additional records from illustrative examples or
supporting narrative details.

4. Project objectives

Within Section 2, “Proposed objective(s)”, extract each principal
project-objective statement represented by the source.

The scope consists of the objective to promote sustainable-land-
management mechanisms, the objective concerning technical innovations,
ecosystem functions and cross-sectoral mechanisms, and the objective
concerning multi-sectoral land and watershed planning with local
participation.

5. Project components

Within Section 3, “Preliminary description”, extract one record for
each explicitly labelled project component.

For each component:

- preserve the component number and component name;
- preserve a concise source-grounded description of the component;
- extract the explicitly represented estimated cost as Value;
- preserve the represented monetary scale in Unit;
- preserve any explicit approximation, ceiling or threshold wording
  associated with the cost in Qualifier.

Do not calculate a total component cost.

6. Safeguard policies

Within Section 4, “Safeguard Policies that Might Apply”, extract:

- each explicitly listed safeguard-policy item;
- the currently assessed environmental-assessment category;
- the potential environmental-assessment category described for
  community sub-project grants with environmental implications.

Preserve source codes, labels and represented spellings exactly as
shown. Do not silently repair typographical errors or category codes.

7. Tentative financing

Within Section 5, “Tentative financing”, extract:

- one record for every explicitly represented financing-source row;
- the explicitly represented Total row.

Preserve the source labels exactly as represented.

Use the represented monetary scale as Unit.

Do not calculate or recompute the Total.

8. Contact information

Within Section 6, “Contact point”, extract one record for each labelled
contact field:

- Contact
- Title
- Tel
- Fax
- Email

Preserve phone numbers and email addresses as JSON strings.

For every included record extract exactly these fields:

- Category
- Topic
- Description
- Value
- Unit
- Qualifier
- Reporting Period
- Source Location

Category:

Use exactly one of:

- Project metadata
- Development issue
- Bank rationale
- Project objective
- Project component
- Safeguard policy
- Financing
- Contact information

Topic:

- Preserve the relevant source-grounded project field, issue,
  objective, component, policy, financing source or contact item.
- Do not merge distinct source observations.

Description:

- Provide a concise source-grounded description of the represented
  record.
- Do not add external interpretation.

Value:

- Use a JSON number for explicitly represented numeric values.
- Use a JSON string for explicitly represented textual values, codes,
  dates, telephone numbers, email addresses, textual fractions or
  category labels.
- Use null when no separate Value is represented.
- Preserve textual fractional quantities in their represented textual
  form rather than converting them to numeric percentages.
- Do not derive separate numbers from percentages embedded within a
  complete textual field.
- Do not calculate, infer, derive, rescale or convert values.

Unit:

- Preserve the explicitly associated measurement unit or scale.
- Use null when no explicit unit applies.
- Do not place approximation, threshold or inequality wording in Unit.

Qualifier:

- Preserve explicit source qualifiers or modifiers associated with a
  Value in this field.
- This includes approximation, threshold, extent or ceiling wording
  represented by the source.
- Use null when no explicit qualifier applies.
- Do not merge Qualifier wording into Unit.

Reporting Period:

- Preserve explicitly associated dates or periods.
- Use null when no separate reporting period is explicitly associated
  with the record.

Source Location:

Use concise physical-DOC locations grounded in the source-page
boundaries exposed by the converted representation, for example:

- DOC page 1 — Header
- DOC page 1 — Section 1, Key Development Issues
- DOC page 2 — Section 1, Key Development Issues
- DOC page 2 — Section 1, Rationale for Bank Involvement
- DOC page 3 — Section 2, Proposed objective(s)
- DOC page 3 — Section 3, Preliminary description
- DOC page 4 — Section 4, Safeguard Policies that Might Apply
- DOC page 4 — Section 5, Tentative financing
- DOC page 4 — Section 6, Contact point

Additional extraction rules:

- Use only information explicitly represented in the attached source representation.
- Preserve source wording, codes, labels and spellings where relevant.
- Preserve represented typographical errors rather than silently correcting them.
- Preserve repeated observations if the fixed scope explicitly requires them in distinct source locations.
- Do not use external knowledge.
- Do not follow external links.
- Do not calculate or infer missing information.
- Do not repair source values or codes.
- Do not convert units.
- Do not add explanatory examples from narrative text outside the defined extraction scope.
- Ignore Markdown syntax and page-boundary labels except as structural cues.
- Verify that all content within the defined source scope has been processed.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names and field order defined below.

Expected JSON structure:

{
  "document_id": "D8",
  "branch": "B",
  "records": [
    {
      "Category": null,
      "Topic": null,
      "Description": null,
      "Value": null,
      "Unit": null,
      "Qualifier": null,
      "Reporting Period": null,
      "Source Location": null
    }
  ]
}

Return only the JSON object.
""".strip()

PROMPT_PATH.write_text(
    BRANCH_B_PROMPT,
    encoding="utf-8"
)

PROMPT_SHA256 = sha256_file(PROMPT_PATH)

print("Prompt saved:", PROMPT_PATH.name)
print("Prompt SHA-256:", PROMPT_SHA256)
print(BRANCH_B_PROMPT)


In [ ]:
# ============================================================
# 8. Create pre-extraction experiment metadata
# ============================================================

EXPERIMENT_METADATA_PRE = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": SOURCE_PATH.name,
    "source_format": SOURCE_FORMAT,
    "source_sha256": SOURCE_SHA256,
    "source_verified": SOURCE_HASH_MATCH,
    "legacy_binary_word_format": LEGACY_BINARY_DOC,
    "expected_physical_page_count": EXPECTED_PHYSICAL_PAGE_COUNT,
    "observed_rendered_page_count": OBSERVED_PAGE_COUNT,
    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "conversion_method":
        CONVERSION_METHOD,
    "representation_file": STRUCTURAL_MARKDOWN_PATH.name,
    "representation_sha256": STRUCTURAL_MARKDOWN_SHA256,
    "intermediate_pdf_file": INTERMEDIATE_PDF_PATH.name,
    "intermediate_pdf_sha256": INTERMEDIATE_PDF_SHA256,
    "direct_document_ingestion": False,
    "structural_conversion_applied": True,
    "conversion_path": "legacy DOC -> PDF -> page-aware Markdown",
    "conversion_fallback_used": False,
    "complete_source_document_retained": True,
    "scope_filtering_applied": False,
    "page_boundaries_made_explicit": True,
    "diagnostic_text_extraction_applied": True,
    "diagnostic_text_extraction_tool": "antiword",
    "diagnostic_text_used_as_model_input": False,
    "ocr_applied": False,
    "normalisation_applied": False,
    "semantic_rewriting_applied": False,
    "unit_conversion_applied": False,
    "source_spelling_correction_applied": False,
    "manual_reconstruction_applied": False,
    "manual_correction_applied": False,
    "content_validation_performed":
        False,
    "expected_extraction_scope": {
        "expected_record_count": EXPECTED_RECORD_COUNT,
        "expected_category_counts": EXPECTED_CATEGORY_COUNTS,
        "expected_fields": EXPECTED_FIELDS
    },
    "reference_expectations_disclosed_to_model": False,
    "conversion_report_file": CONVERSION_REPORT_PATH.name,
    "conversion_integrity_file": CONVERSION_INTEGRITY_PATH.name,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "prompt_file": PROMPT_PATH.name,
    "prompt_sha256": PROMPT_SHA256,
    "expected_output_format":
        "JSON object with document_id, branch and records",
    "execution_environment": "Independent ChatGPT conversation",
}

EXPERIMENT_METADATA_PRE_PATH.write_text(
    json.dumps(EXPERIMENT_METADATA_PRE, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(json.dumps(EXPERIMENT_METADATA_PRE, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 9. Download model-input artefacts
# ============================================================

for path in [
    STRUCTURAL_MARKDOWN_PATH,
    CONVERSION_REPORT_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_METADATA_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH
]:
    files.download(path)

print(
    "\nIndependent extraction instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D8_branch_B_structural_markdown.md.\n"
    "3. Submit the exact D8_branch_B_prompt.txt content once.\n"
    "4. Do not upload the original DOC, Stage 1 reference values, "
    "Validation A outputs, or expected record/category counts.\n"
    "5. Preserve the complete first response exactly as returned in TXT.\n"
    "6. Do not repair, reorder, correct, or regenerate the response."
)


In [ ]:
# ============================================================
# 10. Upload and preserve the untouched Branch B response
# ============================================================

uploaded_output = files.upload()

txt_paths = [
    Path(name)
    for name in uploaded_output
    if name.lower().endswith(".txt")
]

if len(txt_paths) != 1:
    raise ValueError(
        "Upload exactly one TXT file containing the complete D8 Branch B response."
    )

UPLOADED_RAW_RESPONSE_PATH = txt_paths[0]
RAW_RESPONSE_TEXT = UPLOADED_RAW_RESPONSE_PATH.read_text(encoding="utf-8")

if not RAW_RESPONSE_TEXT.strip():
    raise ValueError("The uploaded D8 Branch B response is empty.")

RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)

RAW_RESPONSE_SHA256 = sha256_file(RAW_RESPONSE_PATH)

print("Raw response preserved:", RAW_RESPONSE_PATH.name)
print("Raw response SHA-256:", RAW_RESPONSE_SHA256)


In [ ]:
# ============================================================
# 11. Parse without repairing the response
# ============================================================

valid_json = False
json_parsing_error = None
parsed_response = None

try:
    parsed_response = json.loads(RAW_RESPONSE_TEXT)
    valid_json = True
except json.JSONDecodeError as error:
    json_parsing_error = str(error)

top_level_object_valid = valid_json and isinstance(parsed_response, dict)

document_id_present = (
    top_level_object_valid
    and "document_id" in parsed_response
)
document_id_correct = (
    document_id_present
    and parsed_response.get("document_id") == DOCUMENT_ID
)

branch_present = (
    top_level_object_valid
    and "branch" in parsed_response
)
branch_correct = (
    branch_present
    and parsed_response.get("branch") == BRANCH
)

records_present = (
    top_level_object_valid
    and "records" in parsed_response
)
records_is_list = (
    records_present
    and isinstance(parsed_response.get("records"), list)
)

records_evaluable = all([
    valid_json,
    top_level_object_valid,
    records_present,
    records_is_list
])

extracted_records = (
    parsed_response["records"]
    if records_evaluable
    else []
)

observed_record_count = (
    len(extracted_records)
    if records_evaluable
    else None
)

parsed_extraction_created = False
parsed_extraction_sha256 = None


print("Valid JSON:", valid_json)
print("JSON parsing error:", json_parsing_error)
print("Records evaluable:", records_evaluable)
print("Observed records:", observed_record_count)


In [ ]:
# ============================================================
# 12. Record-schema and field-type checks
# ============================================================

record_structure_issues = []
field_type_issues = []
missing_mandatory_values = []

for record_index, record in enumerate(extracted_records):
    if not isinstance(record, dict):
        record_structure_issues.append({
            "record_index": record_index,
            "issue": "Record is not a JSON object"
        })
        continue

    observed_fields = list(record.keys())

    missing_fields = [
        field for field in EXPECTED_FIELDS
        if field not in record
    ]

    extra_fields = [
        field for field in observed_fields
        if field not in EXPECTED_FIELDS
    ]

    field_order_correct = observed_fields == EXPECTED_FIELDS

    if missing_fields or extra_fields or not field_order_correct:
        record_structure_issues.append({
            "record_index": record_index,
            "missing_fields": missing_fields,
            "extra_fields": extra_fields,
            "field_order_correct": field_order_correct,
            "observed_fields": observed_fields
        })

    for field in STRING_OR_NULL_FIELDS:
        value = record.get(field)

        if value is not None and not isinstance(value, str):
            field_type_issues.append({
                "record_index": record_index,
                "field": field,
                "observed_type": type(value).__name__,
                "expected_type": "string or null"
            })

    value = record.get("Value")

    if (
        isinstance(value, bool)
        or (
            value is not None
            and not isinstance(value, (str, int, float))
        )
    ):
        field_type_issues.append({
            "record_index": record_index,
            "field": "Value",
            "observed_type": type(value).__name__,
            "expected_type": "string, number or null"
        })

    for field in MANDATORY_CONTENT_FIELDS:
        value = record.get(field)

        if value is None or value == "":
            missing_mandatory_values.append({
                "record_index": record_index,
                "field": field
            })

record_schema_valid = (
    len(record_structure_issues) == 0
    if records_evaluable
    else None
)

records_with_type_issues = (
    len({issue["record_index"] for issue in field_type_issues})
    if records_evaluable
    else None
)

field_types_valid = (
    records_with_type_issues == 0
    if records_evaluable
    else None
)

mandatory_fields_complete = (
    len(missing_mandatory_values) == 0
    if records_evaluable
    else None
)

print("Record schema valid:", record_schema_valid)
print("Field types valid:", field_types_valid)
print("Mandatory fields complete:", mandatory_fields_complete)


In [ ]:
# ============================================================
# 13. Content/scope diagnostics — separate from schema validity
# ============================================================

if records_evaluable:

    # --------------------------------------------------------
    # Record-count diagnostics
    # --------------------------------------------------------

    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )


    # --------------------------------------------------------
    # Category diagnostics
    # --------------------------------------------------------

    observed_category_counts = dict(
        Counter(
            record.get("Category")
            for record in extracted_records
            if isinstance(record, dict)
        )
    )


    categories_valid = set(
        observed_category_counts
    ).issubset(
        ALLOWED_CATEGORIES
    )


    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )


    # --------------------------------------------------------
    # Exact complete-record duplicate diagnostic
    # --------------------------------------------------------

    duplicate_counter = Counter(
        tuple(
            json.dumps(
                record.get(field),
                ensure_ascii=False,
                sort_keys=True
            )
            for field in EXPECTED_FIELDS
        )

        for record
        in extracted_records

        if isinstance(record, dict)
    )


    duplicate_records = [
        list(key)

        for key, count
        in duplicate_counter.items()

        if count > 1
    ]


    duplicate_record_count = len(
        duplicate_records
    )


    # --------------------------------------------------------
    # Value-type diagnostics
    # --------------------------------------------------------

    numeric_value_count = sum(
        (
            isinstance(
                record.get("Value"),
                (int, float)
            )
            and not isinstance(
                record.get("Value"),
                bool
            )
        )

        for record
        in extracted_records

        if isinstance(record, dict)
    )


    text_value_count = sum(
        isinstance(
            record.get("Value"),
            str
        )

        for record
        in extracted_records

        if isinstance(record, dict)
    )


    null_value_count = sum(
        record.get("Value") is None

        for record
        in extracted_records

        if isinstance(record, dict)
    )


    # --------------------------------------------------------
    # Qualifier diagnostics
    # --------------------------------------------------------

    observed_qualifier_values = sorted({
        record.get("Qualifier").strip().casefold()

        for record
        in extracted_records

        if (
            isinstance(record, dict)
            and isinstance(
                record.get("Qualifier"),
                str
            )
            and record.get(
                "Qualifier"
            ).strip()
        )
    })

    qualifier_presence = {
        qualifier:
            any(
                qualifier.casefold()
                in observed_qualifier

                for observed_qualifier
                in observed_qualifier_values
            )

        for qualifier
        in EXPECTED_QUALIFIER_VALUES
    }


    expected_qualifiers_preserved = all(
        qualifier_presence.values()
    )

    qualifier_embedded_in_unit_records = [
        {
            "record_index":
                record_index,

            "Unit":
                record.get("Unit")
        }

        for record_index, record
        in enumerate(extracted_records)

        if (
            isinstance(record, dict)
            and isinstance(
                record.get("Unit"),
                str
            )
            and any(
                qualifier.casefold()
                in record.get(
                    "Unit"
                ).casefold()

                for qualifier
                in EXPECTED_QUALIFIER_VALUES
            )
        )
    ]


    # --------------------------------------------------------
    # Textual-quantity preservation
    # --------------------------------------------------------


    extracted_text_values_casefold = {
        record.get("Value").strip().casefold()

        for record
        in extracted_records

        if (
            isinstance(record, dict)
            and isinstance(
                record.get("Value"),
                str
            )
        )
    }


    textual_quantity_checks = {
        "one-quarter":
            "one-quarter"
            in extracted_text_values_casefold,

        "one-third":
            "one-third"
            in extracted_text_values_casefold
    }


    textual_quantities_preserved = all(
        textual_quantity_checks.values()
    )


    # --------------------------------------------------------
    # Source-typo / code fidelity diagnostics
    # --------------------------------------------------------

    extraction_text = json.dumps(
        extracted_records,
        ensure_ascii=False
    )


    source_typo_checks = {
        marker:
            marker in extraction_text

        for marker
        in [
            "Indigenous Pelple",
            "F1",
            "BORROWER/RECEPIENT"
        ]
    }


    source_typos_preserved = all(
        source_typo_checks.values()
    )


else:

    record_count_valid = None

    observed_category_counts = None

    categories_valid = None

    category_counts_valid = None

    duplicate_records = None

    duplicate_record_count = None

    numeric_value_count = None

    text_value_count = None

    null_value_count = None

    observed_qualifier_values = None

    qualifier_presence = None

    expected_qualifiers_preserved = None

    qualifier_embedded_in_unit_records = None

    textual_quantity_checks = None

    textual_quantities_preserved = None

    source_typo_checks = None

    source_typos_preserved = None


# ------------------------------------------------------------
# Content-diagnostic object
# ------------------------------------------------------------

CONTENT_DIAGNOSTICS = {
    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches_reference":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "categories_valid":
        categories_valid,

    "category_counts_match_reference":
        category_counts_valid,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_value_count":
        (
            len(missing_mandatory_values)
            if records_evaluable
            else None
        ),

    "duplicate_complete_record_count":
        duplicate_record_count,

    "numeric_value_count":
        numeric_value_count,

    "text_value_count":
        text_value_count,

    "null_value_count":
        null_value_count,

    "observed_qualifier_values":
        observed_qualifier_values,

    "expected_qualifier_presence":
        qualifier_presence,

    "expected_qualifiers_preserved":
        expected_qualifiers_preserved,

    "qualifier_embedded_in_unit_count":
        (
            len(
                qualifier_embedded_in_unit_records
            )
            if records_evaluable
            else None
        ),

    "textual_quantity_checks":
        textual_quantity_checks,

    "textual_quantities_preserved":
        textual_quantities_preserved,

    "source_typo_checks":
        source_typo_checks,

    "source_typos_preserved":
        source_typos_preserved
}


print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 14. Determine technical/schema validity
# ============================================================

structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    record_schema_valid is True,
    field_types_valid is True
])

parsed_extraction_created = False
parsed_extraction_sha256 = None

if structurally_evaluable:
    PARSED_EXTRACTION = {
        "document_id":
            parsed_response.get("document_id"),
        "branch":
            parsed_response.get("branch"),
        "records":
            extracted_records
    }

    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            PARSED_EXTRACTION,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    parsed_extraction_created = True
    parsed_extraction_sha256 = sha256_file(
        PARSED_EXTRACTION_PATH
    )

TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "valid_json":
        bool(valid_json),

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        bool(top_level_object_valid),

    "document_id_present":
        bool(document_id_present),

    "document_id_correct":
        bool(document_id_correct),

    "branch_present":
        bool(branch_present),

    "branch_correct":
        bool(branch_correct),

    "records_present":
        bool(records_present),

    "records_is_list":
        bool(records_is_list),

    "records_evaluable":
        bool(records_evaluable),

    "record_schema_valid":
        record_schema_valid,

    "record_structure_issues":
        record_structure_issues
        if records_evaluable else None,

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "field_type_issues":
        field_type_issues
        if records_evaluable else None,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "scope_complete":
        bool(record_count_valid)
        if record_count_valid is not None
        else False,

    "structurally_evaluable":
        bool(structurally_evaluable)
}

TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR
    / "D8_branch_B_technical_diagnostics.json"
)

TECHNICAL_DIAGNOSTICS_PATH.write_text(
    json.dumps(TECHNICAL_DIAGNOSTICS, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(json.dumps(TECHNICAL_DIAGNOSTICS, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 15. Final metadata and experiment summary
# ============================================================

EXPERIMENT_METADATA = {
    **EXPERIMENT_METADATA_PRE,
    "raw_response_file": RAW_RESPONSE_PATH.name,
    "raw_response_sha256": RAW_RESPONSE_SHA256,
    "parsed_extraction_file":
        PARSED_EXTRACTION_PATH.name if parsed_extraction_created else None,
    "parsed_extraction_sha256": parsed_extraction_sha256,
    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,
    "structurally_evaluable":
        bool(structurally_evaluable),
    "content_validation_performed":
        False,
    "json_valid": valid_json,
    "records_evaluable": records_evaluable,
    "observed_record_count": observed_record_count,
    "observed_category_counts": observed_category_counts,
    "notes": (
        "Branch B uses a complete page-aware structural Markdown "
        "representation derived from the original four-page legacy DOC. "
        "The source is rendered to PDF to recover physical page boundaries, "
        "then converted to Markdown with pymupdf4llm. No content is filtered "
        "by extraction scope. Stage 1 reference values and expected counts "
        "are not supplied to the model. Content-level validation is performed "
        "separately in Validation B — D8."
    )
}

EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(EXPERIMENT_METADATA, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

EXPERIMENT_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": SOURCE_PATH.name,
    "source_sha256": SOURCE_SHA256,
    "source_verified": SOURCE_HASH_MATCH,
    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "conversion_method":
        CONVERSION_METHOD,
    "conversion_path": "legacy DOC -> PDF -> page-aware Markdown",
    "representation_file": STRUCTURAL_MARKDOWN_PATH.name,
    "representation_sha256": STRUCTURAL_MARKDOWN_SHA256,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "structural_conversion_applied": True,
    "complete_source_document_retained": True,
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "observed_record_count": observed_record_count,
    "record_count_matches": record_count_valid,
    "expected_category_counts": EXPECTED_CATEGORY_COUNTS,
    "observed_category_counts": observed_category_counts,
    "category_counts_match": category_counts_valid,
    "valid_json": valid_json,
    "records_evaluable": records_evaluable,
    "record_schema_valid": record_schema_valid,
    "field_types_valid": field_types_valid,
    "structurally_evaluable": bool(structurally_evaluable),
    "scope_complete": record_count_valid,
    "duplicate_complete_record_count": duplicate_record_count,
    "expected_qualifiers_preserved": expected_qualifiers_preserved,
    "textual_quantities_preserved": textual_quantities_preserved,
    "source_typos_preserved": source_typos_preserved,
    "parsed_extraction_created": bool(structurally_evaluable),
    "content_validation_performed":
        False,

    "notes": (
        "Content-level validation is performed separately "
        "in Validation B — D8."
    )
}

EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(EXPERIMENT_SUMMARY, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(json.dumps(EXPERIMENT_SUMMARY, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 16. Final artefact inventory and downloads
# ============================================================

GENERATED_OUTPUTS = [
    STRUCTURAL_MARKDOWN_PATH,
    CONVERSION_REPORT_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_METADATA_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]

if parsed_extraction_created:
    GENERATED_OUTPUTS.append(PARSED_EXTRACTION_PATH)

print("Generated D8 Branch B files:")

for path in GENERATED_OUTPUTS:
    print("-", path.name, "| exists:", path.exists())

for path in GENERATED_OUTPUTS:
    files.download(path)
